In [65]:
import pandas as pd
import numpy as np

df = pd.read_csv('DA_clean.csv')

df.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales,Item_Category,Outlet_Years
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380,Food,14
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228,Drinks,4
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700,Food,14
3,FDX07,19.20,Regular,0.073719,Fruits and Vegetables,182.0950,OUT010,1998,Small,Tier 3,Grocery Store,732.3800,Food,15
4,NCD19,8.93,Non-Edible,0.064963,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052,Non-Consumable,26


CLASSIF TARGET

In [66]:
def sales_category(x):
    if x < 2000:
        return 'Low'
    elif x < 5000:
        return 'Medium'
    else:
        return 'High'

df['Sales_Category'] = df['Item_Outlet_Sales'].apply(sales_category)

In [67]:
# cek hasil
df[['Item_Outlet_Sales', 'Sales_Category']].head()

,Item_Outlet_Sales,Sales_Category
0,3735.1380,Medium
1,443.4228,Low
2,2097.2700,Medium
3,732.3800,Low
4,994.7052,Low


In [68]:
# cek distribusi
df['Sales_Category'].value_counts()

,count
Sales_Category,
Low,4667
Medium,3227
High,629


FEATURE SELECTION

In [69]:
# target
y = df['Sales_Category']

In [70]:
drop_cols = [
    'Sales_Category',
    'Item_Outlet_Sales',
    'Item_Identifier',
    'Outlet_Identifier',
    'Outlet_Establishment_Year'
]

X = df.drop(columns=drop_cols)

ENCODING

In [71]:
# ONE HOT
X = pd.get_dummies(X, drop_first=True)

X.head()

,Item_Weight,Item_Visibility,Item_MRP,Outlet_Years,Item_Fat_Content_Non-Edible,Item_Fat_Content_Regular,Item_Type_Breads,Item_Type_Breakfast,Item_Type_Canned,Item_Type_Dairy,...,Item_Type_Starchy Foods,Outlet_Size_Medium,Outlet_Size_Small,Outlet_Location_Type_Tier 2,Outlet_Location_Type_Tier 3,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3,Item_Category_Food,Item_Category_Non-Consumable
0,9.30,0.016047,249.8092,14,False,False,False,False,False,True,...,False,True,False,False,False,True,False,False,True,False
1,5.92,0.019278,48.2692,4,False,True,False,False,False,False,...,False,True,False,False,True,False,True,False,False,False
2,17.50,0.016760,141.6180,14,False,False,False,False,False,False,...,False,True,False,False,False,True,False,False,True,False
3,19.20,0.073719,182.0950,15,False,True,False,False,False,False,...,False,False,True,False,True,False,False,False,True,False
4,8.93,0.064963,53.8614,26,True,False,False,False,False,False,...,False,False,False,False,True,True,False,False,False,True


SCALING

In [72]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [73]:
# Encode y
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(y)

In [74]:
#Train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

SMOTE

In [75]:
from imblearn.over_sampling import SMOTE

In [76]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

pd.Series(y_train_smote).value_counts()

,count
2,3706
1,3706
0,3706


LOGISTIC REGRESSION

In [77]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_smote, y_train_smote)

LogisticRegression(max_iter=1000)

RANDOM FOREST CLASSIFIER

In [78]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf_model.fit(X_train_smote, y_train_smote)

RandomForestClassifier(n_estimators=300, random_state=42)

XGBOOST CLASSIFIER

In [79]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

xgb_model.fit(X_train_smote, y_train_smote)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

EVAL

In [80]:
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

pred = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

print(classification_report(y_test, pred))

Accuracy: 0.706158357771261
              precision    recall  f1-score   support

           0       0.37      0.48      0.42       106
           1       0.83      0.77      0.80       961
           2       0.61      0.65      0.63       638

    accuracy                           0.71      1705
   macro avg       0.60      0.63      0.61      1705
weighted avg       0.72      0.71      0.71      1705

